# Category 8 — Business Insights
### Total Questions: 2

---

1. Write a SQL query to display year on year growth for each product
2. Find the percentage contribution of each product to total revenue

⬆️ Upload sql_practice.db from your computer

In [ ]:
# ⬆️ Upload sql_practice.db from your computer
from google.colab import files
uploaded = files.upload()
print("✅ Database uploaded!")

In [ ]:
import sqlite3
import pandas as pd
import os

_cwd = os.getcwd()
_db = os.path.join(_cwd, "sql_practice.db")
if not os.path.isfile(_db):
    _db = os.path.join(os.path.dirname(_cwd), "sql_practice.db")
conn = sqlite3.connect(_db)

# Verify tables loaded
pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)


## 1. Year on Year Growth for Each Product

**Difficulty:** 🔴 Hard

**Subtopic:** Window Functions / LAG + DATE + Growth Calculation

---

**Table: products**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| transaction_id   | int     |
| product_id       | int     |
| transaction_date | date    |
| spend            | int     |
+------------------+---------+
```

- `transaction_id` represents the unique transaction.
- `product_id` represents the product.
- `transaction_date` represents when the transaction happened.
- `spend` represents the amount spent in that transaction.

---

**Problem**

Write a SQL query to display the **year on year growth** for each product.

Output should have `year`, `product_id` and `yoy_growth`.

YoY Growth formula:
```
yoy_growth = ((current_year_spend - prev_year_spend) / prev_year_spend) * 100
```

**Example 1**

**Input**

products table:
```
+----------------+------------+------------------+-------+
| transaction_id | product_id | transaction_date | spend |
+----------------+------------+------------------+-------+
| 1              | 101        | 2022-01-01       | 1000  |
| 2              | 101        | 2022-06-01       | 1500  |
| 3              | 101        | 2023-01-01       | 1800  |
| 4              | 101        | 2023-06-01       | 2000  |
| 5              | 102        | 2022-03-01       | 500   |
| 6              | 102        | 2023-03-01       | 800   |
| 7              | 102        | 2024-03-01       | 1200  |
+----------------+------------+------------------+-------+
```

**Output**
```
+------+------------+------------+
| year | product_id | yoy_growth |
+------+------------+------------+
| 2023 | 101        | 85.71      |
| 2024 | 102        | 50.00      |
+------+------------+------------+
```

In [ ]:
# 👀 Preview the table
print("products table:")
pd.read_sql_query("SELECT * FROM products", conn)

In [ ]:
# ✅ Solution
query = """
WITH yearly AS (
    SELECT
        product_id,
        STRFTIME('%Y', transaction_date) AS year,
        SUM(spend) AS total_spend
    FROM products
    GROUP BY product_id, year
),
yoy AS (
    SELECT
        year,
        product_id,
        total_spend,
        LAG(total_spend) OVER (PARTITION BY product_id ORDER BY year) AS prev_spend
    FROM yearly
)
SELECT
    year,
    product_id,
    ROUND((total_spend - prev_spend) * 100.0 / prev_spend, 2) AS yoy_growth
FROM yoy
WHERE prev_spend IS NOT NULL
"""

pd.read_sql_query(query, conn)

**Explanation**

- **CTE 1 — yearly:** Groups transactions by `product_id` and `year` using `STRFTIME('%Y', transaction_date)` to extract the year. Calculates `SUM(spend)` per product per year.
- **CTE 2 — yoy:** Uses `LAG(total_spend)` to fetch the **previous year's spend** for each product using `PARTITION BY product_id ORDER BY year`.
- **Final SELECT:** Applies the YoY growth formula — `(current - previous) / previous * 100`.
- `WHERE prev_spend IS NOT NULL` excludes the first year of each product since it has no previous year to compare.
- `STRFTIME('%Y', date)` is SQLite syntax — in MySQL use `YEAR(date)`.

> 💡 YoY Growth > 0 means growth, < 0 means decline, = 0 means no change.

## 2. Percentage Contribution of Each Product to Total Revenue

**Difficulty:** 🔴 Hard

**Subtopic:** Window Functions / SUM OVER + Arithmetic + CTE

---

**Table: products**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| transaction_id   | int     |
| product_id       | int     |
| transaction_date | date    |
| spend            | int     |
+------------------+---------+
```

- `transaction_id` represents the unique transaction.
- `product_id` represents the product.
- `spend` represents the amount spent in that transaction.

---

**Problem**

Write a SQL query to find the **percentage contribution** of each product's total spend to the **overall total revenue**.

Return `product_id`, `total_spend` and `revenue_percentage`.

**Example 1**

**Input**

products table:
```
+----------------+------------+------------------+-------+
| transaction_id | product_id | transaction_date | spend |
+----------------+------------+------------------+-------+
| 1              | 101        | 2022-01-01       | 1000  |
| 2              | 101        | 2022-06-01       | 1500  |
| 3              | 101        | 2023-01-01       | 1800  |
| 4              | 101        | 2023-06-01       | 2000  |
| 5              | 102        | 2022-03-01       | 500   |
| 6              | 102        | 2023-03-01       | 800   |
| 7              | 102        | 2024-03-01       | 1200  |
+----------------+------------+------------------+-------+
```

**Output**
```
+------------+-------------+--------------------+
| product_id | total_spend | revenue_percentage |
+------------+-------------+--------------------+
| 101        | 6300        | 67.74              |
| 102        | 2500        | 26.88              |
+------------+-------------+--------------------+
```

In [ ]:
# 👀 Preview the table
print("products table:")
pd.read_sql_query("SELECT * FROM products", conn)

In [ ]:
# ✅ Solution
query = """
WITH product_totals AS (
    SELECT
        product_id,
        SUM(spend) AS total_spend
    FROM products
    GROUP BY product_id
)
SELECT
    product_id,
    total_spend,
    ROUND(total_spend * 100.0 / SUM(total_spend) OVER (), 2) AS revenue_percentage
FROM product_totals
"""

pd.read_sql_query(query, conn)

**Explanation**

- **CTE — product_totals:** Groups all transactions by `product_id` and calculates `SUM(spend)` for each product.
- `SUM(total_spend) OVER ()` — the empty `OVER()` means the window covers **the entire result set** — giving us the grand total revenue across all products.
- `total_spend * 100.0 / SUM(total_spend) OVER ()` divides each product's total by the grand total and multiplies by 100 to get the percentage.
- `ROUND(..., 2)` rounds to 2 decimal places.
- We use `100.0` instead of `100` to force **decimal division** in SQLite.

> 💡 `SUM() OVER ()` with empty parentheses is a very powerful pattern — it gives you the grand total on every row without collapsing the result like `GROUP BY` would.